# Kaya Identity와 LMDI 분해분석

## 국가와 기간을 바꾼 CO₂ 변화 분해

강의와 다른 국가 또는 기간을 선택해 배출량 변화를 네 요인의 기여로 나누고 완전분해 여부를 검증합니다.


## 이번 실습에서 완성할 것

- 두 endpoint와 네 양수 요인을 확인합니다.
- 기여도 부호와 크기를 비교합니다.
- 기여도 합과 관측 변화량의 closure를 확인합니다.

**완료 결과:** 기여도 표·그림, 완전분해 확인값, 인과표현을 사용하지 않은 결과문장


## 이번 교시에서 사용할 데이터

### Kaya–LMDI 수업용 국가 snapshot

| 항목 | 내용 |
|---|---|
| 구성 | Germany, Japan, South Korea, United States |
| 기간과 규모 | 1990~2024년, 140행 |
| 한 행의 의미 | 한 국가의 한 연도 CO₂·경제·에너지 관측 |
| 원자료 열 | `co2_tonnes`, `co2_tonnes_per_capita`, `gdp_per_capita_intl2021`, `primary_energy_twh` |
| 수업에서 계산할 값 | 인구, 총 GDP, 에너지집약도, 탄소집약도 |

이 파일은 Global Carbon Budget/OWID의 CO₂, World Bank/OWID의 1인당 GDP, U.S. EIA의 1차에너지 소비량을 같은 국가와 기간에 맞춘 검증 snapshot이며 원 논문의 분석자료가 아닙니다. EIA 에너지는 TJ를 TWh로 변환했고, 독일 1990년은 EIA 동독·서독 계열을 합산했습니다. 인구는 총배출량과 1인당 배출량의 비로 복원하고, 총 GDP와 두 집약도는 해당 값을 이용해 계산합니다.

강의에서는 South Korea의 2000~2022년을 사용합니다. 적용 실습에서는 국가 또는 시작·종료연도를 하나만 변경해 결과 차이를 비교합니다. LMDI의 로그비를 계산하려면 선택한 두 시점의 모든 요인이 양수여야 합니다.


## 이번 실습의 분석 순서

기준 사례 `JPN, 2000→2022`의 결과를 보존합니다. 이어서 국가 또는 기간 가운데 하나만 바꾸고, 기준·변경 사례의 관측 변화량과 네 기여도를 한 표에서 비교합니다.

**남길 결과:** 기준–변경 비교표, 현재 사례의 기여도 그림, 두 사례의 closure 확인, 인과표현을 사용하지 않은 결과문장


## 실습 구간 안내

| 구분 | 수행 범위 |
|---|---|
| **수업 중 필수** | 기준–변경 사례 계산, 기여도 그림, closure 검사 |
| **시간이 남으면** | 여러 국가·기간의 기여도 민감성 |
| **수업 후 확장(선택)** | 세 기간에서 기여도 부호 안정성 |

수업 중에는 필수 구간의 결과와 마지막 `PASS`를 먼저 완성합니다. 확장 구간을 실행하지 않아도 필수 셀이 독립적으로 작동하도록 구성되어 있습니다.


## 실행 환경


In [ ]:
from pathlib import Path
import platform
import warnings

import numpy as np
import pandas as pd
from IPython.display import display

pd.set_option("display.max_columns", 40)
pd.set_option("display.width", 130)
warnings.filterwarnings("ignore", category=FutureWarning)

def find_course_data(relative_path: str) -> Path:
    """0교시에서 준비한 data 폴더의 파일을 찾습니다."""
    relative = Path(relative_path)
    checked = []
    for start in [Path.cwd(), *Path.cwd().parents, Path("/content")]:
        candidate = (start / "data" / relative).resolve()
        if candidate in checked:
            continue
        checked.append(candidate)
        if candidate.exists():
            return candidate
    raise FileNotFoundError(
        f"데이터 파일을 찾지 못했습니다: {relative_path}. "
        "과정 폴더의 00_setup.ipynb를 먼저 실행해 주세요."
    )
print("Python:", platform.python_version())
print("pandas:", pd.__version__)


## 1. 입력자료와 파생요인 검증

CO₂, 인구, GDP, 에너지를 같은 국가–연도 key로 읽고 단위를 확인합니다. 인구와 GDP total은 공개 지표의 관계에서 복원합니다.


In [ ]:
LMDI_PATH = find_course_data("methods/kaya_lmdi_sample.csv")
kaya = pd.read_csv(LMDI_PATH)
analysis_name = "Kaya-LMDI sensitivity analysis"

print("shape:", kaya.shape)
print("countries:", sorted(kaya["code"].unique()))
print("year range:", kaya["year"].min(), kaya["year"].max())
display(kaya.head(3))


## 2. Additive LMDI의 입력과 출력

로그평균 가중치를 사용해 시작–종료 CO₂ 변화량을 인구·풍요·에너지집약도·탄소집약도 기여로 나눕니다.


In [ ]:
def log_mean(x1: float, x0: float) -> float:
    if np.isclose(x1, x0):
        return float(x1)
    return float((x1 - x0) / (np.log(x1) - np.log(x0)))

def additive_lmdi(frame: pd.DataFrame, code: str, start_year: int, end_year: int):
    country = frame.loc[frame["code"].eq(code)].sort_values("year").copy()
    endpoints = country.loc[country["year"].isin([start_year, end_year])].set_index("year")
    if set(endpoints.index) != {start_year, end_year}:
        raise ValueError("선택한 시작·종료 연도가 데이터에 모두 있어야 합니다.")

    endpoints["population"] = endpoints["co2_tonnes"] / endpoints["co2_tonnes_per_capita"]
    endpoints["gdp_total"] = endpoints["population"] * endpoints["gdp_per_capita_intl2021"]
    endpoints["energy_intensity"] = endpoints["primary_energy_twh"] / endpoints["gdp_total"]
    endpoints["carbon_intensity"] = endpoints["co2_tonnes"] / endpoints["primary_energy_twh"]

    factor_columns = {
        "population": "population effect",
        "gdp_per_capita_intl2021": "affluence effect",
        "energy_intensity": "energy-intensity effect",
        "carbon_intensity": "carbon-intensity effect",
    }
    c0 = float(endpoints.loc[start_year, "co2_tonnes"])
    c1 = float(endpoints.loc[end_year, "co2_tonnes"])
    weight = log_mean(c1, c0)
    effects = []
    for column, label in factor_columns.items():
        f0 = float(endpoints.loc[start_year, column])
        f1 = float(endpoints.loc[end_year, column])
        effects.append({"effect": label, "contribution_tonnes": weight * np.log(f1 / f0)})
    result = pd.DataFrame(effects)
    return endpoints.reset_index(), result, c1 - c0


## 3. 국가와 분석기간 설정

설정한 국가·기간의 endpoint만 사용해 기여도를 계산합니다. 중간 연도 추세는 별도 분석 대상입니다.


In [ ]:
# 기준 사례는 그대로 보존합니다. 아래 CURRENT 세 값 가운데 하나만 바꾸어 비교합니다.
BASELINE_COUNTRY_CODE = "JPN"
BASELINE_START_YEAR = 2000
BASELINE_END_YEAR = 2022

COUNTRY_CODE = "JPN"  # 예: KOR
START_YEAR = 2000      # 또는 시작연도만 변경
END_YEAR = 2022

baseline_endpoints, baseline_contributions, baseline_observed_change = additive_lmdi(
    kaya, BASELINE_COUNTRY_CODE, BASELINE_START_YEAR, BASELINE_END_YEAR
)
endpoints, contributions, observed_change = additive_lmdi(
    kaya, COUNTRY_CODE, START_YEAR, END_YEAR
)

baseline_result_table = baseline_contributions.assign(
    contribution_mt=lambda d: d["contribution_tonnes"] / 1_000_000
)
result_table = contributions.assign(
    contribution_mt=lambda d: d["contribution_tonnes"] / 1_000_000
)

case_settings = pd.DataFrame([
    {
        "case": "기준",
        "country": BASELINE_COUNTRY_CODE,
        "start_year": BASELINE_START_YEAR,
        "end_year": BASELINE_END_YEAR,
        "observed_change_Mt": baseline_observed_change / 1_000_000,
    },
    {
        "case": "현재 설정",
        "country": COUNTRY_CODE,
        "start_year": START_YEAR,
        "end_year": END_YEAR,
        "observed_change_Mt": observed_change / 1_000_000,
    },
])

contribution_comparison = (
    baseline_result_table.set_index("effect")[["contribution_mt"]]
    .rename(columns={"contribution_mt": "baseline_Mt"})
    .join(
        result_table.set_index("effect")[["contribution_mt"]]
        .rename(columns={"contribution_mt": "current_Mt"})
    )
)
contribution_comparison["current_minus_baseline_Mt"] = (
    contribution_comparison["current_Mt"]
    - contribution_comparison["baseline_Mt"]
)

print("[비교할 두 사례의 설정과 관측 CO2 변화]")
display(case_settings.round(3))
print("[기준 사례를 보존한 요인별 기여도 비교]")
display(contribution_comparison.round(3))

print("\n[현재 설정의 시작값과 종료값]")
display(endpoints[["year", "co2_tonnes", "population", "gdp_total", "primary_energy_twh"]])
display(result_table[["effect", "contribution_mt"]])
print("observed CO2 change (Mt):", observed_change / 1_000_000)
print("sum of effects (Mt):", result_table["contribution_mt"].sum())

factor_names = [
    "population", "gdp_per_capita_intl2021", "energy_intensity", "carbon_intensity"
]
factor_view = endpoints.set_index("year")[factor_names].T
factor_view.columns = [str(START_YEAR), str(END_YEAR)]
factor_view["end/start ratio"] = factor_view[str(END_YEAR)] / factor_view[str(START_YEAR)]
print("\n[현재 설정의 요인 방향: ratio가 1보다 크면 증가, 작으면 감소]")
display(factor_view)

closure_view = pd.DataFrame({
    "값": ["실제 CO2 변화", "네 기여도의 합", "둘의 차이"],
    "Mt CO2": [
        observed_change / 1_000_000,
        result_table["contribution_mt"].sum(),
        observed_change / 1_000_000 - result_table["contribution_mt"].sum(),
    ],
})
print("[현재 설정의 완전분해 확인]")
display(closure_view)

baseline_closure_error = float(
    baseline_result_table["contribution_tonnes"].sum() - baseline_observed_change
)
assert abs(baseline_closure_error) < 1e-4


## 4. 기여도와 Closure 검증

막대 방향은 변화량을 늘린 요인과 줄인 요인을 나타냅니다. 합계가 실제 변화량과 일치하는지 closure error로 확인합니다.


In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style="whitegrid", context="notebook")
colors = ["#c44e52" if value > 0 else "#4c72b0" for value in result_table["contribution_mt"]]
fig, ax = plt.subplots(figsize=(9, 4.5))
ax.barh(result_table["effect"], result_table["contribution_mt"], color=colors)
ax.axvline(0, color="black", linewidth=0.8)
ax.set(title=f"Additive LMDI | {COUNTRY_CODE}, {START_YEAR}–{END_YEAR}", xlabel="Contribution to CO2 change (Mt)")
plt.tight_layout()
plt.show()

closure_error = float(result_table["contribution_tonnes"].sum() - observed_change)
checks = pd.Series({
    "two endpoints": len(endpoints) == 2,
    "positive factor values": (endpoints.select_dtypes("number").drop(columns="year") > 0).all().all(),
    "four effects": len(result_table) == 4,
    "complete decomposition": abs(closure_error) < 1e-4,
}, name="passed")
display(checks.to_frame())
assert checks.all()
print("LMDI 분해 검증: PASS")


## 시간이 남으면 — 국가와 기간에 따른 기여도 민감성

먼저 같은 2000~2022년의 독일·일본·한국·미국을 비교합니다. 이어서 한국의 1990~2010과 2010~2022를 비교합니다. 실행 전에 각 사례에서 가장 큰 양의 기여도와 음의 기여도를 예상해 기록하세요.


In [ ]:
comparison_cases = [
    ("DEU", 2000, 2022), ("JPN", 2000, 2022),
    ("KOR", 2000, 2022), ("USA", 2000, 2022),
    ("KOR", 1990, 2010), ("KOR", 2010, 2022),
]
comparison_rows = []
for code, start_year, end_year in comparison_cases:
    case_endpoints, case_contributions, case_change = additive_lmdi(kaya, code, start_year, end_year)
    case_row = {"case": f"{code} {start_year}–{end_year}", "observed_change_Mt": case_change / 1_000_000}
    case_row.update({
        row.effect: row.contribution_tonnes / 1_000_000
        for row in case_contributions.itertuples(index=False)
    })
    case_row["closure_residual_Mt"] = (
        case_contributions["contribution_tonnes"].sum() - case_change
    ) / 1_000_000
    comparison_rows.append(case_row)

lmdi_comparison = pd.DataFrame(comparison_rows).set_index("case")
display(lmdi_comparison.round(3))
assert lmdi_comparison["closure_residual_Mt"].abs().max() < 1e-8
print("모든 사례의 closure: PASS")


## 수업 후 확장(선택) — 세 기간에서 기여도 부호가 유지되는가

필수 실습을 마친 뒤 수행합니다. 국가 하나와 서로 다른 세 기간을 직접 선택합니다. 각 Kaya 요인의 LMDI 기여도가 세 기간에서 계속 증가 방향인지, 감소 방향인지, 기간에 따라 부호가 바뀌는지 확인합니다.


In [ ]:
STABILITY_COUNTRY = "KOR"
STABILITY_WINDOWS = [(1990, 2000), (2000, 2010), (2010, 2022)]

assert len(STABILITY_WINDOWS) == 3, "서로 다른 세 기간을 입력하세요."
available_years = set(kaya.loc[kaya["code"].eq(STABILITY_COUNTRY), "year"].astype(int))
for start_year, end_year in STABILITY_WINDOWS:
    assert start_year < end_year, "각 기간은 시작연도보다 종료연도가 뒤여야 합니다."
    assert {start_year, end_year}.issubset(available_years), "선택한 endpoint가 데이터에 없습니다."

stability_rows = []
for start_year, end_year in STABILITY_WINDOWS:
    _, contributions, observed_change = additive_lmdi(
        kaya, STABILITY_COUNTRY, start_year, end_year
    )
    for row in contributions.itertuples(index=False):
        contribution_mt = row.contribution_tonnes / 1_000_000
        sign = "increase" if contribution_mt > 1e-9 else "decrease" if contribution_mt < -1e-9 else "near zero"
        stability_rows.append({
            "period": f"{start_year}–{end_year}",
            "effect": row.effect,
            "contribution_Mt": contribution_mt,
            "sign": sign,
            "observed_change_Mt": observed_change / 1_000_000,
        })

stability_long = pd.DataFrame(stability_rows)
contribution_table = stability_long.pivot(index="effect", columns="period", values="contribution_Mt")
sign_table = stability_long.pivot(index="effect", columns="period", values="sign")
sign_stability = sign_table.nunique(axis=1).eq(1).rename("same sign in all three periods")

print("[세 기간의 요인별 기여도: Mt CO2]")
display(contribution_table.round(3))
print("[세 기간의 기여 방향]")
display(sign_table.join(sign_stability))
print("부호가 유지되지 않는 요인:", sign_stability.index[~sign_stability].tolist())


## 결과 기록과 ChatGPT 검토

다음 셀은 기준–현재 설정표, 기여도 비교표, closure 표와 필수 검사를 직접 넣습니다.

- **관찰:** 네 기여도의 부호·크기와 기준 대비 변화
- **조건:** 국가, 시작·종료연도, 가법형 LMDI와 단위
- **한계:** 산술적 기여와 정책 인과효과의 구분


In [ ]:
review_prompt = f"""
Kaya–LMDI 결과를 다음 실제 출력만 사용해 검토해 주세요.

- 기준과 현재 설정:
{case_settings.to_string(index=False)}
- 기준 대비 요인별 기여도(Mt CO2):
{contribution_comparison.to_string()}
- 현재 설정 closure 표:
{closure_view.to_string(index=False)}
- 필수 검사:
{checks.to_string()}

관찰 | 분석조건 | 해석한계의 세 항목으로 답해 주세요. 네 기여도의 부호와
크기, 기여도 합과 관측 변화량을 실제 값으로 인용하고, 산술적 기여를 정책의
인과효과라고 표현하지 마세요.
""".strip()
print("[ChatGPT에 복사할 교시별 검토 프롬프트]\n")
print(review_prompt)


## 완료 확인

마지막 출력이 `LMDI 분해 검증: PASS`인지 확인해 주세요. 이 검사는 필요한 계산이 실행됐는지 확인할 뿐, 연구 주장의 타당성을 자동 판정하지 않습니다.


## 결과를 정리할 질문

- 어떤 국가 또는 기간을 바꾸었으며 네 기여도의 부호와 크기는 어떻게 달라졌습니까?
- 기여도 합과 관측 변화량의 차이는 허용오차 안에 있습니까?
- 이 분해가 제시하는 산술적 기여와 별도 연구설계가 필요한 인과효과를 구분했습니까?
